DECORATORS

In [6]:
# decorator is the function that takes function as argument and returns function
# the main use of this decorators are without changing the original function using decorators we can get the desired output

# without arguments
def my_decorator(func):
    def wrapper():
        print("This is wrapper function")
        func()
    return wrapper

# manual decorator
# here my_decorator is decorator
# greet is the function that we are passing to the my_decorator(function)
# returns a function wrapper which catches call
# this is manual representation of decorator

def greet():
    print("Hi")

call=my_decorator(greet)


# @ decorator
# this is the same as above
@my_decorator
def greeting():
    print("hello") 


# calling the decorators
call()
greeting()


This is wrapper function
Hi
This is wrapper function
hello


In [8]:
# handling arguments
def my_decorator(func):
    def wrapper(*args,**kwArgs):
        result=func(*args,**kwArgs)
        return result
    return wrapper

@my_decorator
def add(a,b):
    print(a+b)

add(3,4)


7


In [16]:
# functools @wraps


# problem

def my_decorator(func):
    def wrapper(*args,**kwArgs):
        '''Hi this is wrapper'''
        result=func(*args,**kwArgs)
        return result
    return wrapper

@my_decorator
def add(a,b):
    '''Hi this is add'''
    print(a+b)

add(3,4)
#help(add)
print(add.__name__) # returns wrapper != add
print(add.__doc__)  # returns wrapper doc != add doc
print()

# solution 
# using @wraps we can reach until add()
from functools import wraps
def my_decorator(func):
    @wraps(func)
    def wrapper(*args,**kwArgs):
        '''Hi this is wrapper'''
        result=func(*args,**kwArgs)
        return result
    return wrapper

@my_decorator
def add(a,b):
    '''Hi this is add'''
    print(a+b)

add(3,4)
#help(add)
print(add.__name__) # returns add
print(add.__doc__)  # returns  add doc

7
wrapper
Hi this is wrapper

7
add
Hi this is add


In [21]:
# decorators with arguments
def decorator(arg):
    def my_decorator(func):
        def wrapper(*args,**kwArgs):
            result=func(*args,**kwArgs)
            print(f"this is {arg}")
            return result
        return wrapper
    return my_decorator


@decorator("info")
def info_message(msg):
    print(f"{msg} from info message")


@decorator("error")
def error_message(msg):
    print(f"{msg} from error message")

info_message("this is info")
error_message("this is error")

this is info from info message
this is info
this is error from error message
this is error


In [31]:
# class decorator
# A. a decorator appplied to a class

def announce(cls):  
    print(f"{cls.__name__} has been created")
    return cls


@announce   # function taking class as argument and returning class
class Person:  
    pass

# here we don't have to call class like function
# decorator will be called when class was created(Person)


# adding method to a class
def announce(cls):
    def bake(self):
        print("Hi this is mahesh")
    cls.bake=bake # adding method to a class
    return cls
    
@announce
class Person:
    pass

p=Person()
p.bake()


# adding __repr__

def announce(cls):
    def __repr__(self):
        print(f"{cls.__name__} : {self.__dict__}")
    cls.__repr__=__repr__
    return cls

@announce
class Person:
    def __init__(self,name,age):
        self.name=name
        self.age=age
print(Person("mahesh",25))

Person has been created
Hi this is mahesh
Person : {'name': 'mahesh', 'age': 25}


TypeError: __str__ returned non-string (type NoneType)

In [38]:
# keeping the registry
registry=[]
def keep_registry(cls):
    registry.append(cls)
    return cls

@keep_registry
class Person:
    pass

@keep_registry
class Person1:
    pass

@keep_registry
class Person2:
    pass

print(registry)

[<class '__main__.Person'>, <class '__main__.Person1'>, <class '__main__.Person2'>]


In [48]:
# class used as a decorator
from functools import update_wrapper
class Employee:
    def __init__(self,func):
        self.func=func
        update_wrapper(self,func) # this is similar to @wraps in functions
    def __call__(self,*args,**kwArgs):
        result=self.func(*args,**kwArgs)
        print(f"{args} employee has been created in database")
        return result

@Employee
def add_employee(name):
    print("Hi")


add_employee("mahesh")
# for every method call __call__() method will be called


Hi
('mahesh',) employee has been created in database


CONTEXT_MANAGERS

In [ ]:
# closing file problem
f=open("file.txt")
data=f.read()
f.close()
# if read() raises any error f.close() never calls, file stays open
# there are 2 ways to solve this problem

# 1.try/finally
f=open("file.txt")
try:
    data=f.read()
finally:
    f.close()


# 2.with
with open("file.txt") as f:
    f.read()
# here using with calls close() function automatically even if read() function raises error
# with calls 2 special methods __enter__() and __exit__()

# checking how thse 2 methods works
class my_context:
    def __enter__(self):
        print("entering") 
    def __exit__(self,exc_type,exc_value,traceback):
        print("leaving")

with my_context() as f:
    print("execute this block")

print("--------------")

class my_context:
    def __enter__(self):
        print("entering") 
    def __exit__(self,exc_type,exc_value,traceback):
        '''
        return False    # (or None) → the exception keeps propagating — normal
        return True      # → the exception is swallowed, silently
        '''
        if exc_type==None:
            print("leaving")
        else:
            print(f"{exc_type}:{exc_value}")

with my_context() as f:
    print("execute this block")
    raise ValueError("oops")

entering
execute this block
leaving
--------------
entering
execute this block
<class 'ValueError'>:oops


ValueError: oops